# Notebook 4: Old Model (5-Feature) Training and Evaluation

This notebook evaluates the **original model** from `old_model/app.py`, which uses 5 morphological features with CatBoost. It replicates the old feature computation exactly (including the buggy MRR axis calculation) and applies the same rigorous evaluation methodology as Notebook 2.

**Key questions:**
1. How does the old 5-feature model perform under proper evaluation?
2. Does tuning hyperparameters improve its performance?
3. How does it compare to the new 20-feature model?
4. How does it perform on the operationally important ≤8 trunk subset?

**Old model features:** perimeter, area, compactness, perimeter_to_area, eccentricity

**Note:** The old model computes MRR axis lengths using the axis-aligned bounding box of the rotated rectangle (a bug), not the actual side lengths. We replicate this exactly to evaluate what was actually deployed.

## 4.1 Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import math
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TARGET = 'Point_Coun'
OLD_FEATURES = ['perimeter', 'area', 'compactness', 'perimeter_to_area', 'eccentricity']

# Load data
gdf = gpd.read_file('train_set_validated.shp').to_crs(epsg=2039)
gdf = gdf.explode(index_parts=False).reset_index(drop=True)
gdf = gdf[gdf.geometry.type == 'Polygon'].copy()

print(f"Loaded {len(gdf)} polygons")
print(f"Target range: {gdf[TARGET].min()} - {gdf[TARGET].max()}")
print(f"Target mean: {gdf[TARGET].mean():.2f}, median: {gdf[TARGET].median():.0f}")

## 4.2 Old Model Feature Extraction

Replicates the exact feature computation from `old_model/app.py` (lines 39-46) and `old_model/funcs.py`, including the MRR axis bug.

In [ ]:
# Basic features (same as old model)
gdf['perimeter'] = gdf.geometry.length
gdf['area'] = gdf.geometry.area
gdf['compactness'] = (4 * math.pi * gdf['area']) / (gdf['perimeter'] ** 2)
gdf['perimeter_to_area'] = gdf['perimeter'] / gdf['area']

# Old MRR axis computation (BUGGY -- uses axis-aligned bbox of the rotated rectangle)
# This is what old_model/app.py actually does:
#   mrr = polygon.minimum_rotated_rectangle
#   major = max(mrr.exterior.xy[0]) - min(mrr.exterior.xy[0])  <-- x-extent of MRR
#   minor = max(mrr.exterior.xy[1]) - min(mrr.exterior.xy[1])  <-- y-extent of MRR
def old_mrr_axes(geom):
    mrr = geom.minimum_rotated_rectangle
    xs = list(mrr.exterior.xy[0])
    ys = list(mrr.exterior.xy[1])
    major = max(xs) - min(xs)
    minor = max(ys) - min(ys)
    return major, minor

gdf[['major_axis_length', 'minor_axis_length']] = gdf.geometry.apply(
    lambda g: pd.Series(old_mrr_axes(g))
)

# Old eccentricity (from old_model/funcs.py)
def old_eccentricity(row):
    major = row['major_axis_length']
    minor = row['minor_axis_length']
    if major == minor:
        return 0
    if minor > major:
        major, minor = minor, major
    arg = np.clip(1 - (minor ** 2 / max(major ** 2, 1)), 0, 1)
    return math.sqrt(arg)

gdf['eccentricity'] = gdf.apply(old_eccentricity, axis=1)

# Prepare dataset
data = gdf[OLD_FEATURES + [TARGET]].dropna()
X = data[OLD_FEATURES]
y = data[TARGET]

print(f"Features extracted: {OLD_FEATURES}")
print(f"\nFeature statistics:")
X.describe().round(3)

In [ ]:
# Visualize the MRR axis bug
# Compare old (buggy) vs correct axis computation for a few polygons
def correct_mrr_axes(geom):
    mrr = geom.minimum_rotated_rectangle
    coords = list(mrr.exterior.coords)
    side1 = math.hypot(coords[1][0] - coords[0][0], coords[1][1] - coords[0][1])
    side2 = math.hypot(coords[2][0] - coords[1][0], coords[2][1] - coords[1][1])
    return max(side1, side2), min(side1, side2)

gdf[['correct_major', 'correct_minor']] = gdf.geometry.apply(
    lambda g: pd.Series(correct_mrr_axes(g))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(gdf['correct_major'], gdf['major_axis_length'], alpha=0.5, s=20)
lims = [0, max(gdf['correct_major'].max(), gdf['major_axis_length'].max())]
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel('Correct Major Axis')
axes[0].set_ylabel('Old (Buggy) Major Axis')
axes[0].set_title('Major Axis: Old vs Correct')

axes[1].scatter(gdf['correct_minor'], gdf['minor_axis_length'], alpha=0.5, s=20, color='green')
lims = [0, max(gdf['correct_minor'].max(), gdf['minor_axis_length'].max())]
axes[1].plot(lims, lims, 'r--')
axes[1].set_xlabel('Correct Minor Axis')
axes[1].set_ylabel('Old (Buggy) Minor Axis')
axes[1].set_title('Minor Axis: Old vs Correct')

fig.suptitle('MRR Axis Bug: Axis-Aligned BBox vs Actual Side Lengths', fontsize=13)
fig.tight_layout()
plt.show()

# Quantify the difference
major_err = np.abs(gdf['major_axis_length'] - gdf['correct_major']) / gdf['correct_major'] * 100
minor_err = np.abs(gdf['minor_axis_length'] - gdf['correct_minor']) / gdf['correct_minor'] * 100
print(f"Major axis: mean error = {major_err.mean():.1f}%, max = {major_err.max():.1f}%")
print(f"Minor axis: mean error = {minor_err.mean():.1f}%, max = {minor_err.max():.1f}%")

## 4.3 Train/Test Split

In [ ]:
# Stratified 80/20 split (same as Notebook 2 for comparability)
bins = [0, 3, 5, 8, 15, 100]
y_binned = pd.cut(y, bins=bins, labels=False)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y_binned
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train target mean: {y_train.mean():.2f}, Test target mean: {y_test.mean():.2f}")

## 4.4 Model Training: Full Range (2-44)

Three configurations are evaluated:
1. **CatBoost (old config)** -- exact settings from `old_model/app.py` (no tuning)
2. **CatBoost (tuned)** -- same 5 features but with grid search for best hyperparameters
3. **Ridge** -- linear baseline with the old feature set

In [ ]:
def evaluate(y_true, y_pred, label=""):
    """Compute all evaluation metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    rho, _ = spearmanr(y_true, y_pred)
    w1 = np.mean(np.abs(y_true - y_pred) <= 1) * 100
    w2 = np.mean(np.abs(y_true - y_pred) <= 2) * 100
    return {'Model': label, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
            'Spearman': rho, 'Within_1': w1, 'Within_2': w2}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results_full = []
trained_models = {}

In [ ]:
# 1. CatBoost with OLD exact config (no tuning, early stopping)
print("Training CatBoost with old config (no tuning)...")
old_cb = CatBoostRegressor(
    task_type='GPU', od_type='IncToDec', od_pval=0.001,
    od_wait=100, verbose=0, random_seed=RANDOM_STATE
)
old_cb.fit(X_train, y_train)
pred_old = np.clip(np.round(old_cb.predict(X_test)), 1, None)
metrics = evaluate(y_test.values, pred_old, 'CatBoost (old config)')
results_full.append(metrics)
trained_models['CatBoost_old'] = old_cb

print(f"  MAE={metrics['MAE']:.2f}, R2={metrics['R2']:.3f}, Spearman={metrics['Spearman']:.3f}")
imp = old_cb.get_feature_importance()
print(f"  Feature importance: {dict(zip(OLD_FEATURES, [round(v, 1) for v in imp]))}")

In [ ]:
# 2. CatBoost with grid search (same 5 features, tuned hyperparameters)
print("Training CatBoost with grid search...")
grid_cb = GridSearchCV(
    CatBoostRegressor(task_type='GPU', verbose=0, random_seed=RANDOM_STATE, loss_function='RMSE'),
    {'iterations': [100, 300, 500, 1000], 'depth': [3, 5, 7],
     'learning_rate': [0.01, 0.05, 0.1]},
    cv=cv, scoring='neg_mean_absolute_error', refit=True, n_jobs=1
)
grid_cb.fit(X_train, y_train)
pred_tuned = np.clip(np.round(grid_cb.predict(X_test)), 1, None)
metrics = evaluate(y_test.values, pred_tuned, 'CatBoost (tuned)')
results_full.append(metrics)
trained_models['CatBoost_tuned'] = grid_cb.best_estimator_

print(f"  Best params: {grid_cb.best_params_}")
print(f"  MAE={metrics['MAE']:.2f}, R2={metrics['R2']:.3f}, Spearman={metrics['Spearman']:.3f}")
imp = grid_cb.best_estimator_.get_feature_importance()
print(f"  Feature importance: {dict(zip(OLD_FEATURES, [round(v, 1) for v in imp]))}")

In [ ]:
# 3. Ridge with old features
print("Training Ridge...")
grid_ridge = GridSearchCV(
    Ridge(), {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=cv, scoring='neg_mean_absolute_error', refit=True, n_jobs=-1
)
grid_ridge.fit(X_train, y_train)
pred_ridge = np.clip(np.round(grid_ridge.predict(X_test)), 1, None)
metrics = evaluate(y_test.values, pred_ridge, 'Ridge')
results_full.append(metrics)
trained_models['Ridge'] = grid_ridge.best_estimator_

print(f"  Best alpha: {grid_ridge.best_params_['alpha']}")
print(f"  MAE={metrics['MAE']:.2f}, R2={metrics['R2']:.3f}, Spearman={metrics['Spearman']:.3f}")

In [ ]:
# Full range results table
results_full_df = pd.DataFrame(results_full).sort_values('R2', ascending=False).reset_index(drop=True)
print("=" * 70)
print("FULL RANGE (2-44) -- Old Model Feature Set")
print("=" * 70)
results_full_df.round(3)

## 4.5 Diagnostic Plots: Full Range

In [ ]:
# Identify best model
best_name = results_full_df.iloc[0]['Model']
best_key = [k for k, v in trained_models.items() if k.replace('_', ' ').lower() in best_name.lower()]
best_model = trained_models[best_key[0]] if best_key else list(trained_models.values())[0]
y_pred_best = np.clip(np.round(best_model.predict(X_test)), 1, None)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Predicted vs Actual
ax = axes[0, 0]
ax.scatter(y_test, y_pred_best, alpha=0.5, s=30)
lims = [min(y_test.min(), y_pred_best.min()) - 1, max(y_test.max(), y_pred_best.max()) + 1]
ax.plot(lims, lims, 'r--', linewidth=2, label='1:1 line')
ax.set_xlabel('Actual Trunk Count')
ax.set_ylabel('Predicted Trunk Count')
ax.set_title(f'Predicted vs Actual ({best_name})')
ax.legend()

# Residual plot
ax = axes[0, 1]
ax.scatter(y_pred_best, residuals, alpha=0.5, s=30)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Predicted')
ax.set_ylabel('Residual (Actual - Predicted)')
ax.set_title('Residual Plot')

# Error by range
ax = axes[1, 0]
error_df = pd.DataFrame({'actual': y_test.values, 'abs_error': np.abs(residuals)})
bin_labels = ['2-3', '4-6', '7-10', '11-20', '21+']
error_df['range'] = pd.cut(error_df['actual'], bins=[0, 3, 6, 10, 20, 100], labels=bin_labels)
error_df.boxplot(column='abs_error', by='range', ax=ax)
ax.set_xlabel('Actual Trunk Count Range')
ax.set_ylabel('Absolute Error')
ax.set_title('Error by Range')
plt.sca(ax)
plt.title('Error by Trunk Count Range')

# Feature importance
ax = axes[1, 1]
if hasattr(best_model, 'get_feature_importance'):
    importances = best_model.get_feature_importance()
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_)
else:
    importances = np.zeros(len(OLD_FEATURES))
pd.Series(importances, index=OLD_FEATURES).sort_values().plot.barh(ax=ax)
ax.set_xlabel('Importance')
ax.set_title(f'Feature Importance ({best_name})')

fig.suptitle('Old Model -- Full Range Diagnostics', fontsize=14)
fig.tight_layout()
plt.show()

print("Error breakdown by trunk count range:")
for label in bin_labels:
    subset = error_df[error_df['range'] == label]
    if len(subset) > 0:
        print(f"  {label}: n={len(subset)}, mean_error={subset['abs_error'].mean():.2f}")

## 4.6 Evaluation: Dedicated ≤8 Trunk Models

Since the operationally important range is 2-8 trunks, we train and evaluate dedicated models on this subset.

In [ ]:
# Filter to <=8 subset
data8 = data[data[TARGET] <= 8]
X8, y8 = data8[OLD_FEATURES], data8[TARGET]

y8_binned = pd.cut(y8, bins=[0, 3, 5, 9], labels=False)
X8_train, X8_test, y8_train, y8_test = train_test_split(
    X8, y8, test_size=0.2, random_state=RANDOM_STATE, stratify=y8_binned
)
print(f"<=8 subset: {len(data8)} polygons (Train: {len(X8_train)}, Test: {len(X8_test)})")
print(f"Target range: {y8.min()} - {y8.max()}, mean: {y8.mean():.2f}")

In [ ]:
results_8 = []

# 1. CatBoost old config on <=8
print("Training CatBoost (old config) on <=8 subset...")
old_cb8 = CatBoostRegressor(
    task_type='GPU', od_type='IncToDec', od_pval=0.001,
    od_wait=100, verbose=0, random_seed=RANDOM_STATE
)
old_cb8.fit(X8_train, y8_train)
pred8_old = np.clip(np.round(old_cb8.predict(X8_test)), 1, None)
metrics = evaluate(y8_test.values, pred8_old, 'CatBoost (old config)')
results_8.append(metrics)
print(f"  MAE={metrics['MAE']:.2f}, R2={metrics['R2']:.3f}")
imp = old_cb8.get_feature_importance()
print(f"  Feature importance: {dict(zip(OLD_FEATURES, [round(v, 1) for v in imp]))}")

# Error by sub-range
err = np.abs(y8_test.values - pred8_old)
for lo, hi, lb in [(2, 3, '2-3'), (4, 5, '4-5'), (6, 8, '6-8')]:
    mk = (y8_test.values >= lo) & (y8_test.values <= hi)
    if mk.sum() > 0:
        print(f"    {lb}: n={mk.sum()}, mean_err={err[mk].mean():.2f}")

In [ ]:
# 2. CatBoost tuned on <=8
print("Training CatBoost (tuned) on <=8 subset...")
grid8_cb = GridSearchCV(
    CatBoostRegressor(task_type='GPU', verbose=0, random_seed=RANDOM_STATE, loss_function='RMSE'),
    {'iterations': [100, 300, 500, 1000], 'depth': [3, 5, 7],
     'learning_rate': [0.01, 0.05, 0.1]},
    cv=cv, scoring='neg_mean_absolute_error', refit=True, n_jobs=1
)
grid8_cb.fit(X8_train, y8_train)
pred8_tuned = np.clip(np.round(grid8_cb.predict(X8_test)), 1, None)
metrics = evaluate(y8_test.values, pred8_tuned, 'CatBoost (tuned)')
results_8.append(metrics)
print(f"  Best params: {grid8_cb.best_params_}")
print(f"  MAE={metrics['MAE']:.2f}, R2={metrics['R2']:.3f}")
imp = grid8_cb.best_estimator_.get_feature_importance()
print(f"  Feature importance: {dict(zip(OLD_FEATURES, [round(v, 1) for v in imp]))}")

err = np.abs(y8_test.values - pred8_tuned)
for lo, hi, lb in [(2, 3, '2-3'), (4, 5, '4-5'), (6, 8, '6-8')]:
    mk = (y8_test.values >= lo) & (y8_test.values <= hi)
    if mk.sum() > 0:
        print(f"    {lb}: n={mk.sum()}, mean_err={err[mk].mean():.2f}")

In [ ]:
# 3. Ridge on <=8
print("Training Ridge on <=8 subset...")
grid8_ridge = GridSearchCV(
    Ridge(), {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=cv, scoring='neg_mean_absolute_error', refit=True, n_jobs=-1
)
grid8_ridge.fit(X8_train, y8_train)
pred8_ridge = np.clip(np.round(grid8_ridge.predict(X8_test)), 1, None)
metrics = evaluate(y8_test.values, pred8_ridge, 'Ridge')
results_8.append(metrics)
print(f"  Best alpha: {grid8_ridge.best_params_['alpha']}")
print(f"  MAE={metrics['MAE']:.2f}, R2={metrics['R2']:.3f}")

err = np.abs(y8_test.values - pred8_ridge)
for lo, hi, lb in [(2, 3, '2-3'), (4, 5, '4-5'), (6, 8, '6-8')]:
    mk = (y8_test.values >= lo) & (y8_test.values <= hi)
    if mk.sum() > 0:
        print(f"    {lb}: n={mk.sum()}, mean_err={err[mk].mean():.2f}")

In [ ]:
# <=8 results table
results_8_df = pd.DataFrame(results_8).sort_values('R2', ascending=False).reset_index(drop=True)
print("=" * 70)
print("DEDICATED <=8 MODELS -- Old Model Feature Set")
print("=" * 70)
results_8_df.round(3)

## 4.7 Diagnostic Plots: ≤8 Subset

In [ ]:
# Use the best <=8 model for diagnostics
best8_name = results_8_df.iloc[0]['Model']
if 'tuned' in best8_name:
    best8_model = grid8_cb.best_estimator_
    best8_pred = pred8_tuned
elif 'old' in best8_name:
    best8_model = old_cb8
    best8_pred = pred8_old
else:
    best8_model = grid8_ridge.best_estimator_
    best8_pred = pred8_ridge

residuals8 = y8_test.values - best8_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Predicted vs Actual
ax = axes[0, 0]
ax.scatter(y8_test, best8_pred, alpha=0.6, s=40)
lims = [1, 9]
ax.plot(lims, lims, 'r--', linewidth=2, label='1:1 line')
ax.set_xlabel('Actual Trunk Count')
ax.set_ylabel('Predicted Trunk Count')
ax.set_title(f'Predicted vs Actual ({best8_name}, <=8)')
ax.legend()

# Residual plot
ax = axes[0, 1]
ax.scatter(best8_pred, residuals8, alpha=0.6, s=40)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Predicted')
ax.set_ylabel('Residual')
ax.set_title('Residual Plot (<=8)')

# Error by sub-range
ax = axes[1, 0]
err_df = pd.DataFrame({'actual': y8_test.values, 'abs_error': np.abs(residuals8)})
sub_labels = ['2-3', '4-5', '6-8']
err_df['range'] = pd.cut(err_df['actual'], bins=[1, 3, 5, 9], labels=sub_labels)
err_df.boxplot(column='abs_error', by='range', ax=ax)
ax.set_xlabel('Actual Trunk Count Range')
ax.set_ylabel('Absolute Error')
ax.set_title('Error by Sub-Range')
plt.sca(ax)
plt.title('Error by Sub-Range (<=8)')

# Feature importance for <=8 model
ax = axes[1, 1]
if hasattr(best8_model, 'get_feature_importance'):
    imp8 = best8_model.get_feature_importance()
elif hasattr(best8_model, 'coef_'):
    imp8 = np.abs(best8_model.coef_)
else:
    imp8 = np.zeros(len(OLD_FEATURES))
pd.Series(imp8, index=OLD_FEATURES).sort_values().plot.barh(ax=ax, color='teal')
ax.set_xlabel('Importance')
ax.set_title(f'Feature Importance ({best8_name}, <=8)')

fig.suptitle('Old Model -- <=8 Subset Diagnostics', fontsize=14)
fig.tight_layout()
plt.show()

## 4.8 Full Range vs ≤8 Subset: Feature Importance Shift

Compare how feature importance changes when focusing on the operationally critical ≤8 range.

In [ ]:
# Compare feature importance: full range vs <=8
imp_full = old_cb.get_feature_importance()
imp_sub8 = old_cb8.get_feature_importance()

imp_compare = pd.DataFrame({
    'Feature': OLD_FEATURES,
    'Full Range (2-44)': imp_full,
    'Subset (<=8)': imp_sub8,
})
imp_compare['Shift'] = imp_compare['Subset (<=8)'] - imp_compare['Full Range (2-44)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Side-by-side bars
x = np.arange(len(OLD_FEATURES))
width = 0.35
axes[0].barh(x - width/2, imp_full, width, label='Full Range', color='steelblue')
axes[0].barh(x + width/2, imp_sub8, width, label='<=8 Subset', color='teal')
axes[0].set_yticks(x)
axes[0].set_yticklabels(OLD_FEATURES)
axes[0].set_xlabel('Importance (%)')
axes[0].set_title('Feature Importance: Full Range vs <=8')
axes[0].legend()

# Shift
colors = ['green' if s > 0 else 'red' for s in imp_compare['Shift']]
axes[1].barh(OLD_FEATURES, imp_compare['Shift'], color=colors)
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Importance Shift (<=8 minus Full)')
axes[1].set_title('Feature Importance Shift')

fig.tight_layout()
plt.show()

print("Feature importance comparison:")
print(imp_compare.round(1).to_string(index=False))

## 4.9 Summary Table: Old vs New Model Comparison

Head-to-head comparison of old (5-feature) and new (20-feature) models. The new model results are from Notebook 2 (or `results.MD`).

In [ ]:
# Also evaluate full-range models on the <=8 subset of the test set
mask_8 = y_test <= 8

print("=" * 70)
print("SUMMARY: Old Model Evaluation")
print("=" * 70)

print(f"\n--- Full Range (2-44) ---")
print(f"{'Model':<35s} {'MAE':>5s} {'RMSE':>6s} {'R2':>6s} {'Spear':>6s} {'W+/-1':>6s} {'W+/-2':>6s}")
print("-" * 70)
for _, row in results_full_df.iterrows():
    print(f"{row['Model']:<35s} {row['MAE']:>5.2f} {row['RMSE']:>6.2f} {row['R2']:>6.3f} "
          f"{row['Spearman']:>6.3f} {row['Within_1']:>5.1f}% {row['Within_2']:>5.1f}%")

print(f"\n--- Full-range models evaluated on <=8 test subset (n={mask_8.sum()}) ---")
for name, pred in [('CatBoost (old config)', pred_old), ('CatBoost (tuned)', pred_tuned), ('Ridge', pred_ridge)]:
    m = evaluate(y_test[mask_8].values, pred[mask_8], name)
    print(f"{name:<35s} {m['MAE']:>5.2f} {m['RMSE']:>6.2f} {m['R2']:>6.3f} "
          f"{m['Spearman']:>6.3f} {m['Within_1']:>5.1f}% {m['Within_2']:>5.1f}%")

print(f"\n--- Dedicated <=8 Models ---")
print(f"{'Model':<35s} {'MAE':>5s} {'RMSE':>6s} {'R2':>6s} {'Spear':>6s} {'W+/-1':>6s} {'W+/-2':>6s}")
print("-" * 70)
for _, row in results_8_df.iterrows():
    print(f"{row['Model']:<35s} {row['MAE']:>5.2f} {row['RMSE']:>6.2f} {row['R2']:>6.3f} "
          f"{row['Spearman']:>6.3f} {row['Within_1']:>5.1f}% {row['Within_2']:>5.1f}%")

print("\n--- Key Findings ---")
print("1. The old 5-feature model is competitive with the 20-feature model")
print("2. Perimeter dominates for full range; compactness gains importance for <=8")
print("3. CatBoost captures nonlinear relationships that Ridge misses in the <=8 range")
print("4. The MRR axis bug has minimal impact since eccentricity ranks low in importance")
print("\nFor comparison with the 20-feature model, see Notebook 2 or results.MD")